In [ ]:
# membandit setup
import os
import sys
from pathlib import Path

# Ensure cleaned_repo (parent of membandit/) is on sys.path when running from notebooks/
_here = Path.cwd().resolve()
for _candidate in [_here, *_here.parents]:
    if (_candidate / "membandit" / "__init__.py").exists():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

# Optional: set MEMORYBANDIT_DATA_DIR to your data root (defaults to /Volumes/T7 Shield)
# os.environ.setdefault("MEMORYBANDIT_DATA_DIR", "/path/to/data/root")

from membandit.plotting_style import set_plot_style
set_plot_style()


In [ ]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

import os
import glob
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from membandit.plotting_style import set_plot_style
from membandit.io import Subject
from membandit._impl.multivariate import run_pooled_alpha_tuning_pipeline, run_smoothed_time_resolved_multivariate_regression_pipeline, run_multivariate_reactivation_pipeline, load_trial_level, load_mv_activation_and_mni, coords_values_to_heatmap_nifti, build_multivariate_csv_path, compute_subject_lrt_map, compute_all_subjects_lrt_maps_from_npz

warnings.filterwarnings('ignore')
set_plot_style()


# Multivariate Time-Resolved Regression + Projection (Template-Style)
This notebook mirrors the structure of `MemoryBandit-TimeResolvedRegression_Template.ipynb`, but uses multivariate ridge encoding at each timepoint and writes projection-ready outputs for downstream MV reactivation analysis.


In [ ]:
# =============================
# Paths + subject discovery
# =============================
base_dir = '/Volumes/T7 Shield'
load_dir = f'{base_dir}/work/qasims01/MemoryBanditData/EMU'

demo_file = f'{base_dir}/projects/guLab/Salman/EMU/MemoryBanditsDatabase.xlsx'
demo_data = pd.read_excel(demo_file, engine='openpyxl')
subj_df = demo_data[(demo_data.LongVersion == 1) & (demo_data.Minerva == 1)]
subj_df = subj_df[~subj_df['MSSMCode'].isin(['MS015', 'MS024', 'MS034', 'MS038'])]

subjects = subj_df.MSSMCode.astype(str).tolist()
print(f'Found {len(subjects)} analyzable subjects')
print(subjects)


In [ ]:
# =============================
# Analysis configuration
# =============================
ROIS = ['AMY', 'HPC', 'OFC', 'dlPFC', 'ACC', 'dmPFC']
PREDICTORS = ['zrpe']
COVARIATES = ['zpm', 'pgamble', 'preward']

EVENT_ENCODING = 'feedback_start'
EVENT_PROJECTION = 'cue_on'
# Use 'combined' when predictor is in combined_df only (e.g. phit, hits); encoding stays feedback_start, trials_gamble align to fb_hfa
REGRESSOR_SOURCE = 'learn'

FREQ_RANGE = (70., 201.)
WIN_MS = 200
DOWNSAMPLE_FACTOR = 5

# Ridge / CV
RIDGE_ALPHA = 1.0
CV_FOLDS = 10
N_PERM = 100
N_SURROGATES = 250
MIN_TRIALS = 10
SEED = 42
ENCODING_METRIC = 'insample_r2'
ENCODING_NULL_TAIL = 'greater'
ENCODING_USE_SURROGATES = True
ENCODING_SURROGATE_NULL = 'feature_shuffle_per_timepoint'

# Target/model mode for encoding
TARGET_MODE = 'auto'
CLASSIFIER_MODEL = 'ridge_classifier'
CLASSIFIER_ALPHA = 1.0

# Pooled alpha tuning (one alpha per subject/ROI/predictor)
RUN_TUNE_ALPHA_POOLED = True
ALPHA_GRID = np.logspace(-3, 3, 13)
ALPHA_DEFAULT = 1.0
ALPHA_SELECTOR = "ridgecv_gcv"
RUN_ALPHA_OUTER_EVAL = True
OUTER_EVAL_FOLDS = 3
ALPHA_TABLE_PATH = f"{load_dir}/mv_alpha_pooled_{EVENT_ENCODING}_{PREDICTORS[0]}.csv"

N_JOBS_OUTER = min(8, os.cpu_count() or 1)
OVERWRITE_CHECKPOINTS = True
OVERWRITE_TRIAL_LEVEL = True
PROJECTION_ALWAYS_USE_SURROGATES = True  # hard-lock projection surrogates

RUN_MV_ENCODING_CSV = True
RUN_MV_PROJECTION_NPZ = True
RUN_SINGLE_SUBJECT_SMOKE = True

BAND = 'hfa'
MV_TRIAL_SAVE_DIR = '/Volumes/T7 Shield/scratch/MemoryBandit/RPE_Reactivation/trial_level/multivariate'

CSV_OUTPUT_PATHS = []
ALPHA_OUTPUT_PATHS = []
ALPHA_TABLES_BY_PREDICTOR = {}
ALPHA_TABLE_ALL = pd.DataFrame()
ALPHA_MAP = {}

SMOKE_SUBJECT = 'MS036' if 'MS036' in subjects else subjects[0]
SMOKE_ROI = 'dmPFC' if 'dmPFC' in ROIS else ROIS[0]
SMOKE_PREDICTOR = PREDICTORS[0]

def is_categorical_predictor(predictor: str) -> bool:
    return str(predictor) == 'rpe_cat'

def get_encoding_runtime_config(predictor: str) -> dict:
    is_cat = is_categorical_predictor(predictor)
    metric = 'cv_balanced_accuracy' if is_cat else ENCODING_METRIC
    target_mode = 'categorical' if is_cat else TARGET_MODE
    return {
        'covariates': COVARIATES,
        'ridge_alpha': float(CLASSIFIER_ALPHA) if is_cat else float(ALPHA_DEFAULT),
        'ridge_alpha_map': None if is_cat else ALPHA_MAP,
        'encoding_metric': metric,
        'target_mode': target_mode,
        'classifier_model': CLASSIFIER_MODEL,
        'classifier_alpha': float(CLASSIFIER_ALPHA),
        'metric_display': metric,
    }

print('Smoke subject:', SMOKE_SUBJECT)
print('Smoke ROI:', SMOKE_ROI)
print('Smoke predictor:', SMOKE_PREDICTOR)


# All-trial LRT configuration
RUN_LRT_ALLTRIAL = True
LRT_ROIS = ['ACC']
LRT_PREDICTORS = PREDICTORS.copy()
LRT_CLUSTER_N_PERM = 1000
LRT_CLUSTER_P_THRESH = 0.05
LRT_CLUSTER_TAIL = 1
LRT_CLUSTER_SEED = 42
LRT_SAVE_DIR = f"{load_dir}/mv_lrt_results"
os.makedirs(LRT_SAVE_DIR, exist_ok=True)


## Pooled Alpha Tuning (per subject x ROI x predictor)

In [ ]:
# # =============================
# # Smoke pooled alpha tuning
# # =============================
# smoke_alpha_df = pd.DataFrame()

# if RUN_TUNE_ALPHA_POOLED:
#     smoke_alpha_df = run_pooled_alpha_tuning_pipeline(
#         subjects=[SMOKE_SUBJECT],
#         rois=[SMOKE_ROI],
#         predictors=[SMOKE_PREDICTOR],
#         base_dir=base_dir,
#         load_dir=load_dir,
#         event=EVENT_ENCODING,
#         covariates=COVARIATES,
#         freq_range=FREQ_RANGE,
#         downsample_factor=DOWNSAMPLE_FACTOR,
#         win_ms=WIN_MS,
#         alpha_grid=ALPHA_GRID,
#                 seed=SEED,
#         min_trials=MIN_TRIALS,
#         default_ridge_alpha=ALPHA_DEFAULT,
#         run_outer_eval=RUN_ALPHA_OUTER_EVAL,
#         outer_eval_folds=OUTER_EVAL_FOLDS,
#         n_jobs_outer=1,
#         verbose=True,
#     )

# print('Smoke pooled-alpha rows:', len(smoke_alpha_df))
# if len(smoke_alpha_df) > 0:
#     print(smoke_alpha_df[['subj','region','predictor','selected_alpha','status','outer_score_mean','outer_score_std']])


In [ ]:
# =============================
# Batch pooled alpha tuning
# =============================
ALPHA_OUTPUT_PATHS = []
ALPHA_TABLES_BY_PREDICTOR = {}

if RUN_TUNE_ALPHA_POOLED:
    for predictor in PREDICTORS:
        if is_categorical_predictor(predictor):
            print(f"Skipping pooled alpha tuning for categorical predictor={predictor}; using CLASSIFIER_ALPHA={CLASSIFIER_ALPHA}.")
            continue
        alpha_df = run_pooled_alpha_tuning_pipeline(
            subjects=subjects,
            rois=ROIS,
            predictors=[predictor],
            base_dir=base_dir,
            load_dir=load_dir,
            event=EVENT_ENCODING,
            covariates=COVARIATES,
            freq_range=FREQ_RANGE,
            downsample_factor=DOWNSAMPLE_FACTOR,
            win_ms=WIN_MS,
            alpha_grid=ALPHA_GRID,
            seed=SEED,
            min_trials=MIN_TRIALS,
            default_ridge_alpha=ALPHA_DEFAULT,
            run_outer_eval=RUN_ALPHA_OUTER_EVAL,
            outer_eval_folds=OUTER_EVAL_FOLDS,
            n_jobs_outer=N_JOBS_OUTER,
            verbose=True,
            regressor_source=REGRESSOR_SOURCE,
        )
        alpha_path = f"{load_dir}/mv_alpha_pooled_{EVENT_ENCODING}_{predictor}.csv"
        alpha_df.to_csv(alpha_path, index=False)
        ALPHA_OUTPUT_PATHS.append(alpha_path)
        ALPHA_TABLES_BY_PREDICTOR[predictor] = alpha_df
        print(f'Saved pooled-alpha table: {alpha_path} (rows={len(alpha_df)})')
else:
    for predictor in PREDICTORS:
        if is_categorical_predictor(predictor):
            print(f"Skipping pooled alpha table load for categorical predictor={predictor}; using CLASSIFIER_ALPHA={CLASSIFIER_ALPHA}.")
            continue
        alpha_path = f"{load_dir}/mv_alpha_pooled_{EVENT_ENCODING}_{predictor}.csv"
        if os.path.exists(alpha_path):
            alpha_df = pd.read_csv(alpha_path)
            ALPHA_OUTPUT_PATHS.append(alpha_path)
            ALPHA_TABLES_BY_PREDICTOR[predictor] = alpha_df
            print(f'Loaded existing pooled-alpha table: {alpha_path} (rows={len(alpha_df)})')
        else:
            print(f'No alpha table found for predictor={predictor}: {alpha_path}')

if len(ALPHA_TABLES_BY_PREDICTOR) > 0:
    ALPHA_TABLE_ALL = pd.concat(ALPHA_TABLES_BY_PREDICTOR.values(), ignore_index=True)
else:
    ALPHA_TABLE_ALL = pd.DataFrame()

print('Alpha table rows (all):', len(ALPHA_TABLE_ALL))


In [ ]:
ALPHA_TABLE_ALL

In [ ]:
# =============================
# Build alpha map for downstream pipelines
# =============================
ALPHA_MAP = {}
if len(ALPHA_TABLE_ALL) > 0:
    for _, row in ALPHA_TABLE_ALL.iterrows():
        key = (str(row['subj']), str(row['region']), str(row['predictor']))
        alpha_val = row.get('selected_alpha', ALPHA_DEFAULT)
        try:
            alpha_val = float(alpha_val)
        except Exception:
            alpha_val = float(ALPHA_DEFAULT)
        if (not np.isfinite(alpha_val)) or (alpha_val <= 0):
            alpha_val = float(ALPHA_DEFAULT)
        ALPHA_MAP[key] = alpha_val

print('Alpha map entries:', len(ALPHA_MAP))
if len(ALPHA_MAP) > 0:
    for k in list(ALPHA_MAP.keys())[:5]:
        print(k, '->', ALPHA_MAP[k])


In [ ]:
# =============================
# Alpha diagnostics
# =============================
if len(ALPHA_TABLE_ALL) == 0:
    print('No alpha tuning results to plot.')
else:
    plot_df = ALPHA_TABLE_ALL.copy()
    plot_df['selected_alpha'] = pd.to_numeric(plot_df['selected_alpha'], errors='coerce')
    plot_df = plot_df[np.isfinite(plot_df['selected_alpha']) & (plot_df['selected_alpha'] > 0)]
    plot_df['log10_alpha'] = np.log10(plot_df['selected_alpha'])

    fig, ax = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
    if len(plot_df) > 0:
        sns.boxplot(data=plot_df, x='region', y='log10_alpha', ax=ax[0], color='lightgray')
        ax[0].set_title('log10(selected alpha) by ROI')
        ax[0].set_xlabel('ROI')
        ax[0].set_ylabel('log10(alpha)')
        ax[0].tick_params(axis='x', rotation=45)

        sns.scatterplot(data=plot_df, x='n_electrodes', y='log10_alpha', hue='region', style='predictor', ax=ax[1], s=45)
        ax[1].set_title('Alpha vs electrode count')
        ax[1].set_xlabel('n_electrodes')
        ax[1].set_ylabel('log10(alpha)')
        ax[1].legend(bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0)
    else:
        ax[0].text(0.5, 0.5, 'No finite alpha values', ha='center', va='center')
        ax[1].text(0.5, 0.5, 'No finite alpha values', ha='center', va='center')
    plt.show()


## Single-subject smoke test (encoding)

In [ ]:
smoke_encoding_df = pd.DataFrame()

if RUN_SINGLE_SUBJECT_SMOKE:
    smoke_cfg = get_encoding_runtime_config(SMOKE_PREDICTOR)
    smoke_encoding_df = run_smoothed_time_resolved_multivariate_regression_pipeline(
        subjects=[SMOKE_SUBJECT],
        event=EVENT_ENCODING,
        predictor=SMOKE_PREDICTOR,
        base_dir=base_dir,
        load_dir=load_dir,
        rois=[SMOKE_ROI],
        freq_range=FREQ_RANGE,
        win_ms=WIN_MS,
        downsample_factor=DOWNSAMPLE_FACTOR,
        n_permutations=N_PERM,
        covariates=smoke_cfg['covariates'],
        min_trials=MIN_TRIALS,
        seed=SEED,
        ridge_alpha=smoke_cfg['ridge_alpha'],
        ridge_alpha_map=smoke_cfg['ridge_alpha_map'],
        predictor_key=SMOKE_PREDICTOR,
        cv_folds=CV_FOLDS,
        encoding_metric=smoke_cfg['encoding_metric'],
        null_tail=ENCODING_NULL_TAIL,
        store_metric_name=True,
        use_surrogates=ENCODING_USE_SURROGATES,
        surrogate_null=ENCODING_SURROGATE_NULL,
        target_mode=smoke_cfg['target_mode'],
        classifier_alpha=smoke_cfg['classifier_alpha'],
        classifier_model=smoke_cfg['classifier_model'],
        n_jobs_outer=1,
        checkpoint_dir=None,
        overwrite_checkpoints=True,
        verbose=True,
        regressor_source=REGRESSOR_SOURCE,
    )

print('Smoke encoding rows:', len(smoke_encoding_df))
if len(smoke_encoding_df) > 0:
    print('Unique timepoints:', smoke_encoding_df['ts'].nunique())
    print('Smoke ridge alpha values:', np.sort(smoke_encoding_df['ridge_alpha'].dropna().unique()))
    if 'metric_name' in smoke_encoding_df.columns:
        print('Smoke metric_name values:', smoke_encoding_df['metric_name'].dropna().unique())
    if 'target_kind' in smoke_encoding_df.columns:
        print('Smoke target_kind values:', smoke_encoding_df['target_kind'].dropna().unique())
    if 'surrogate_null' in smoke_encoding_df.columns:
        print('Smoke surrogate_null values:', smoke_encoding_df['surrogate_null'].dropna().unique())
    print('Smoke metric range:', float(np.nanmin(smoke_encoding_df['metric'])), float(np.nanmax(smoke_encoding_df['metric'])))
    if ENCODING_USE_SURROGATES:
        print('Smoke z_metric range:', float(np.nanmin(smoke_encoding_df['z_metric'])), float(np.nanmax(smoke_encoding_df['z_metric'])))
    print(smoke_encoding_df.head())


In [ ]:
# Ad-hoc test cell removed from main pipeline. Use dedicated smoke and tuning cells instead.

In [ ]:
# Scratch/debug placeholder (intentionally left blank).

In [ ]:
# Scratch/debug placeholder (intentionally left blank).

In [ ]:
# =============================
# Batch multivariate encoding (CSV)
# =============================
mv_encoding_results = {}
required_cols = {
    'predictor', 'metric', 'z_metric', 'raw_p', 'ts', 'subj', 'region',
    'n_trials', 'n_electrodes', 'ridge_alpha', 'cv_folds', 'event', 'covariates'
}

if RUN_MV_ENCODING_CSV:
    alpha_checkpoint_tag = 'pooled' if len(ALPHA_MAP) > 0 else 'fixedalpha'
    for predictor in PREDICTORS:
        enc_cfg = get_encoding_runtime_config(predictor)
        checkpoint_dir = f"{load_dir}/checkpoints_mv_smoothed_{BAND}_{EVENT_ENCODING}_{predictor}_{alpha_checkpoint_tag}"
        df_mv = run_smoothed_time_resolved_multivariate_regression_pipeline(
            subjects=subjects,
            event=EVENT_ENCODING,
            predictor=predictor,
            base_dir=base_dir,
            load_dir=load_dir,
            rois=ROIS,
            freq_range=FREQ_RANGE,
            win_ms=WIN_MS,
            downsample_factor=DOWNSAMPLE_FACTOR,
            n_permutations=N_PERM,
            covariates=enc_cfg['covariates'],
            min_trials=MIN_TRIALS,
            seed=SEED,
            ridge_alpha=enc_cfg['ridge_alpha'],
            ridge_alpha_map=enc_cfg['ridge_alpha_map'],
            predictor_key=predictor,
            cv_folds=CV_FOLDS,
            encoding_metric=enc_cfg['encoding_metric'],
            null_tail=ENCODING_NULL_TAIL,
            store_metric_name=True,
            use_surrogates=ENCODING_USE_SURROGATES,
            surrogate_null=ENCODING_SURROGATE_NULL,
            target_mode=enc_cfg['target_mode'],
            classifier_alpha=enc_cfg['classifier_alpha'],
            classifier_model=enc_cfg['classifier_model'],
            n_jobs_outer=N_JOBS_OUTER,
            checkpoint_dir=checkpoint_dir,
            overwrite_checkpoints=OVERWRITE_CHECKPOINTS,
            verbose=True,
            regressor_source=REGRESSOR_SOURCE,
        )

        missing_cols = sorted(required_cols.difference(df_mv.columns))
        assert not missing_cols, f'Missing required encoding columns: {missing_cols}'
        optional_cols = {
            'metric_name', 'null_tail', 'surrogate_null', 'use_surrogates',
            'target_kind', 'n_classes', 'class_labels_json'
        }
        missing_optional = sorted(optional_cols.difference(df_mv.columns))
        if len(missing_optional) > 0:
            print(f'Warning: missing optional metric metadata columns: {missing_optional}')

        out_csv = build_multivariate_csv_path(
            load_dir=load_dir,
            band=BAND,
            event=EVENT_ENCODING,
            predictor=predictor,
            suffix='smoothed_final',
        )
        df_mv.to_csv(out_csv, index=False)
        CSV_OUTPUT_PATHS.append(out_csv)
        mv_encoding_results[predictor] = df_mv
        print(f'Saved MV encoding CSV: {out_csv} (rows={len(df_mv)})')
else:
    print('RUN_MV_ENCODING_CSV=False, skipping multivariate encoding CSV stage.')


## Smoke encoding plot (single subject)

In [ ]:
# =============================
# Plot smoke encoding (single-subject)
# =============================
from mne.stats import permutation_cluster_1samp_test

roi_order = ['OFC', 'ACC', 'HPC', 'dmPFC', 'dlPFC', 'AMY']
roi_pos = {
    'OFC': (0, 0),
    'ACC': (0, 1),
    'HPC': (0, 2),
    'dmPFC': (1, 0),
    'dlPFC': (1, 1),
    'AMY': (1, 2),
}
roi_colors = {
    'HPC': np.array([0.44314, 0.2902, 1]),
    'AMY': np.array([0.49412, 0, 0.90196]),
    'OFC': np.array([1, 0.75686, 0.41569]),
    'ACC': np.array([0.7098, 0.27059, 0.68235]),
    'dmPFC': np.array([0.40784, 0.01961, 0]),
    'dlPFC': np.array([1, 0.06275, 0.06275]),
}

if len(smoke_encoding_df) == 0:
    print('No smoke encoding rows available. Run the smoke encoding cell first.')
else:
    fig, ax = plt.subplots(2, 3, figsize=(20, 10), sharey=True)
    n_perm_cluster = 2000
    plot_col = 'z_metric' if ENCODING_USE_SURROGATES else 'metric'
    metric_names = smoke_encoding_df['metric_name'].dropna().astype(str).unique() if 'metric_name' in smoke_encoding_df.columns else np.array([])
    is_bal = np.any(metric_names == 'cv_balanced_accuracy')
    y_label = 'z(Balanced Accuracy vs surrogate)' if (ENCODING_USE_SURROGATES and is_bal) else ('z(R² vs surrogate)' if ENCODING_USE_SURROGATES else ('Balanced Accuracy' if is_bal else 'R²'))

    for region in roi_order:
        ix, jx = roi_pos[region]
        color = roi_colors.get(region, np.array([0.3, 0.3, 0.3]))

        region_data = smoke_encoding_df[
            (smoke_encoding_df['predictor'] == SMOKE_PREDICTOR) &
            (smoke_encoding_df['region'] == region)
        ].copy()

        if len(region_data) == 0:
            ax[ix, jx].set_title(f'{region} (no data)')
            ax[ix, jx].axis('off')
            continue

        unique_ts = np.sort(region_data['ts'].unique())
        mean_metric = region_data.groupby('ts')[plot_col].mean().reindex(unique_ts).to_numpy()
        se_metric = region_data.groupby('ts')[plot_col].sem().reindex(unique_ts).to_numpy()

        ax[ix, jx].plot(unique_ts, mean_metric, color=color, linewidth=1.5)
        ax[ix, jx].fill_between(
            unique_ts,
            mean_metric - np.nan_to_num(se_metric, nan=0.0),
            mean_metric + np.nan_to_num(se_metric, nan=0.0),
            color=color,
            alpha=0.2,
        )

        subj_time = region_data.pivot_table(index='subj', columns='ts', values=plot_col, aggfunc='mean')
        subj_time = subj_time.reindex(columns=unique_ts)
        subj_time = subj_time.dropna(axis=0, how='any')

        if subj_time.shape[0] >= 2:
            try:
                T_obs, clusters, cluster_p_values, _ = permutation_cluster_1samp_test(
                    subj_time.to_numpy(dtype=float),
                    n_permutations=n_perm_cluster,
                    tail=0,
                    verbose=False,
                )
                for i_c, c in enumerate(clusters):
                    if cluster_p_values[i_c] <= 0.05:
                        c_ix = c[0]
                        c_ts = unique_ts[c_ix]
                        if len(c_ts) > 0:
                            y0, y1 = ax[ix, jx].get_ylim()
                            band_lo = y0 + 0.88 * (y1 - y0)
                            band_hi = y0 + 0.96 * (y1 - y0)
                            ax[ix, jx].fill_between(c_ts, band_lo, band_hi, color=color, alpha=0.35)
            except Exception as e:
                print(f'Cluster test failed for smoke {region}: {e}')

        if EVENT_ENCODING in ['cue_on', 'choice_time']:
            xticks = unique_ts[::max(1, len(unique_ts)//5)]
        else:
            xticks = unique_ts[::max(1, len(unique_ts)//7)]
        xticklabels = [f'{int(t)}' for t in xticks]

        ax[ix, jx].set(
            xticks=xticks,
            xticklabels=xticklabels,
            xlim=[float(np.min(unique_ts) - 100), float(np.max(unique_ts) + 100)],
            xlabel='time (ms)',
            ylabel=y_label,
            title=f'{region} | smoke'
        )
        ax[ix, jx].axhline(0, color='black', linestyle='--', linewidth=1)
        sns.despine(ax=ax[ix, jx])

        if ix == 0:
            ax[ix, jx].set(xlabel=None, xticklabels=[])
        if jx > 0:
            ax[ix, jx].set(ylabel=None, yticklabels=[])

    plt.suptitle(
        f"Smoke MV encoding (pooled-alpha tuned) | subject={SMOKE_SUBJECT} | predictor={SMOKE_PREDICTOR} | "
        f"metric={('cv_balanced_accuracy' if np.any(smoke_encoding_df.get('metric_name', pd.Series(dtype=str)) == 'cv_balanced_accuracy') else ENCODING_METRIC)} | surrogates={ENCODING_USE_SURROGATES}",
        y=1.02
    )
    plt.tight_layout()
    plt.show()


In [ ]:
# =============================
# Single-subject smoke test (projection)
# =============================
smoke_projection_results = None

if RUN_SINGLE_SUBJECT_SMOKE and RUN_MV_PROJECTION_NPZ:
    assert PROJECTION_ALWAYS_USE_SURROGATES is True
    smoke_projection_results = run_multivariate_reactivation_pipeline(
        subjects=[SMOKE_SUBJECT],
        rois=[SMOKE_ROI],
        predictors=[SMOKE_PREDICTOR],
        base_dir=base_dir,
        load_dir=load_dir,
        event_encoding=EVENT_ENCODING,
        event_projection=EVENT_PROJECTION,
        covariates=COVARIATES,
        freq_range=FREQ_RANGE,
        downsample_factor=DOWNSAMPLE_FACTOR,
        win_ms=WIN_MS,
        min_trials=MIN_TRIALS,
        n_surrogates=N_SURROGATES,
        use_surrogates=True,
        seed=SEED,
        ridge_alpha=ALPHA_DEFAULT,
        ridge_alpha_map=ALPHA_MAP,
        save_trial_level=True,
        trial_save_dir=MV_TRIAL_SAVE_DIR,
        overwrite_trial_level=OVERWRITE_TRIAL_LEVEL,
        trial_save_dtype=np.float32,
        n_jobs_outer=1,
        verbose=True,
        regressor_source=REGRESSOR_SOURCE,
    )

    smoke_npz = os.path.join(
        MV_TRIAL_SAVE_DIR, SMOKE_ROI, f'{SMOKE_SUBJECT}_{SMOKE_PREDICTOR}_zreactivation.npz'
    )
    if not os.path.exists(smoke_npz):
        raise FileNotFoundError(f'Expected smoke NPZ not found: {smoke_npz}')

    z_reactivation_smoke, fb_t_smoke, cue_t_smoke, smoke_meta = load_trial_level(
        SMOKE_ROI, SMOKE_SUBJECT, MV_TRIAL_SAVE_DIR, dependent_var=SMOKE_PREDICTOR
    )

    required_npz_meta = {
        'memory_outcomes', 'trial_index', 'electrode_labels', 'subject', 'roi', 'predictor',
        'event_encoding', 'event_projection', 'n_surrogates', 'ridge_alpha', 'covariates',
        'residualized_predictor', 'activation_patterns', 'z_reactivation_lrt',
        'projection_real_lrt', 'trial_index_lrt', 'memory_outcomes_lrt',
        'n_trials_train', 'n_trials_lrt', 'lrt_axis_mode'
    }
    missing_npz_meta = sorted(required_npz_meta.difference(smoke_meta.keys()))
    assert not missing_npz_meta, f'Missing required NPZ metadata keys: {missing_npz_meta}'
    assert z_reactivation_smoke.ndim == 3, f'Expected 3D z_reactivation, got {z_reactivation_smoke.shape}'
    assert len(smoke_meta['memory_outcomes']) == z_reactivation_smoke.shape[-1], 'memory_outcomes trial mismatch'

    print('Smoke NPZ path:', smoke_npz)
    print('z_reactivation shape:', z_reactivation_smoke.shape)
    print('feedback_times:', len(fb_t_smoke), 'cue_times:', len(cue_t_smoke))
    print('smoke ridge alpha:', float(smoke_meta['ridge_alpha']) if 'ridge_alpha' in smoke_meta else 'NA')
    print('activation_patterns shape:', smoke_meta['activation_patterns'].shape)
    print('meta keys:', sorted(list(smoke_meta.keys()))[:12])
else:
    print('Skipping smoke projection stage.')


In [ ]:
plt.plot(smoke_meta['activation_patterns'][:,0])
plt.plot(smoke_meta['activation_patterns'][:,1])
plt.plot(smoke_meta['activation_patterns'][:,2])
plt.plot(smoke_meta['activation_patterns'][:,3])
plt.plot(smoke_meta['activation_patterns'][:,4])
plt.plot(smoke_meta['activation_patterns'][:,5])
plt.plot(smoke_meta['activation_patterns'][:,6])



In [ ]:
# =============================
# Batch multivariate projection (NPZ)
# =============================
mv_projection_results = None

if RUN_MV_PROJECTION_NPZ:
    assert PROJECTION_ALWAYS_USE_SURROGATES is True
    mv_projection_results = run_multivariate_reactivation_pipeline(
        subjects=subjects,
        rois=ROIS,
        predictors=PREDICTORS,
        base_dir=base_dir,
        load_dir=load_dir,
        event_encoding=EVENT_ENCODING,
        event_projection=EVENT_PROJECTION,
        covariates=COVARIATES,
        freq_range=FREQ_RANGE,
        downsample_factor=DOWNSAMPLE_FACTOR,
        win_ms=WIN_MS,
        min_trials=MIN_TRIALS,
        n_surrogates=N_SURROGATES,
        use_surrogates=True,
        seed=SEED,
        ridge_alpha=ALPHA_DEFAULT,
        ridge_alpha_map=ALPHA_MAP,
        save_trial_level=True,
        trial_save_dir=MV_TRIAL_SAVE_DIR,
        overwrite_trial_level=OVERWRITE_TRIAL_LEVEL,
        trial_save_dtype=np.float32,
        n_jobs_outer=N_JOBS_OUTER,
        verbose=True,
        regressor_source=REGRESSOR_SOURCE,
    )
else:
    print('RUN_MV_PROJECTION_NPZ=False, skipping projection stage.')


In [ ]:
# =============================
# QC plot for one subject/ROI/predictor
# =============================
qc_npz = os.path.join(
    MV_TRIAL_SAVE_DIR, SMOKE_ROI, f'{SMOKE_SUBJECT}_{SMOKE_PREDICTOR}_zreactivation.npz'
)

if os.path.exists(qc_npz):
    z_reactivation, feedback_times, cue_times, meta = load_trial_level(
        SMOKE_ROI, SMOKE_SUBJECT, MV_TRIAL_SAVE_DIR, dependent_var=SMOKE_PREDICTOR
    )
    assert z_reactivation.ndim == 3, f'Expected 3D MV z_reactivation, got shape {z_reactivation.shape}'

    activation_patterns = meta.get('activation_patterns', None)
    if activation_patterns is not None:
        print('activation_patterns shape:', activation_patterns.shape)

    memory_outcomes = np.asarray(meta.get('memory_outcomes', np.array([], dtype=object)))
    all_trials_map = np.nanmean(z_reactivation, axis=2)

    hits_mask = memory_outcomes == 'hits'
    misses_mask = memory_outcomes == 'misses'
    has_hm = (hits_mask.sum() > 0) and (misses_mask.sum() > 0)

    if has_hm:
        hm_map = (
            np.nanmean(z_reactivation[:, :, hits_mask], axis=2) -
            np.nanmean(z_reactivation[:, :, misses_mask], axis=2)
        )

    n_panels = 2 if has_hm else 1
    fig, axes = plt.subplots(1, n_panels, figsize=(6 * n_panels, 5), constrained_layout=True)
    if n_panels == 1:
        axes = [axes]

    extent = [float(cue_times.min()), float(cue_times.max()), float(feedback_times.min()), float(feedback_times.max())]

    vmax_a = np.nanmax(np.abs(all_trials_map))
    if not np.isfinite(vmax_a) or vmax_a == 0:
        vmax_a = 1.0

    im0 = axes[0].imshow(
        all_trials_map,
        origin='lower',
        aspect='auto',
        extent=extent,
        cmap='RdBu_r',
        vmin=-vmax_a,
        vmax=vmax_a,
    )
    axes[0].set_title(f'All trials mean\n{SMOKE_SUBJECT} | {SMOKE_ROI} | {SMOKE_PREDICTOR}')
    axes[0].set_xlabel('Cue Time (s)')
    axes[0].set_ylabel('Feedback Time (s)')
    plt.colorbar(im0, ax=axes[0], shrink=0.8)

    if has_hm:
        vmax_b = np.nanmax(np.abs(hm_map))
        if not np.isfinite(vmax_b) or vmax_b == 0:
            vmax_b = 1.0
        im1 = axes[1].imshow(
            hm_map,
            origin='lower',
            aspect='auto',
            extent=extent,
            cmap='RdBu_r',
            vmin=-vmax_b,
            vmax=vmax_b,
        )
        axes[1].set_title('Hits - Misses')
        axes[1].set_xlabel('Cue Time (s)')
        axes[1].set_ylabel('Feedback Time (s)')
        plt.colorbar(im1, ax=axes[1], shrink=0.8)

    plt.show()

    print('Outcome counts:')
    if memory_outcomes.size > 0:
        print(pd.Series(memory_outcomes).value_counts(dropna=False))
    else:
        print('No memory_outcomes available in metadata.')
else:
    print(f'QC file not found: {qc_npz}')


## All-Trial LRT (4-class memory outcome)


In [ ]:
# =============================
# Smoke LRT map (single subject)
# =============================
smoke_lrt_map = None
smoke_lrt_summary = {}

if RUN_LRT_ALLTRIAL and RUN_SINGLE_SUBJECT_SMOKE:
    z_smoke, fb_t_smoke, cue_t_smoke, meta_smoke = load_trial_level(
        SMOKE_ROI, SMOKE_SUBJECT, MV_TRIAL_SAVE_DIR, dependent_var=SMOKE_PREDICTOR
    )

    if 'z_reactivation_lrt' not in meta_smoke or 'memory_outcomes_lrt' not in meta_smoke:
        raise KeyError('Missing LRT keys in smoke NPZ (z_reactivation_lrt / memory_outcomes_lrt).')

    z_lrt = np.asarray(meta_smoke['z_reactivation_lrt'])
    mem_lrt = np.asarray(meta_smoke['memory_outcomes_lrt'], dtype=object)

    assert z_lrt.ndim == 3, f'Expected 3D z_reactivation_lrt, got {z_lrt.shape}'
    assert len(mem_lrt) == z_lrt.shape[-1], 'memory_outcomes_lrt trial mismatch'

    smoke_lrt_map = compute_subject_lrt_map(z_lrt, mem_lrt, n_jobs=-1, verbose=False)
    smoke_lrt_summary = pd.Series(mem_lrt).value_counts(dropna=False).to_dict()

    print('Smoke LRT map shape:', smoke_lrt_map.shape)
    print('Smoke LRT range:', float(np.nanmin(smoke_lrt_map)), float(np.nanmax(smoke_lrt_map)))
    print('Smoke memory class counts:', smoke_lrt_summary)
else:
    print('Skipping smoke LRT map cell.')


In [ ]:
plt.imshow(
    smoke_lrt_map,
    origin='lower',
    aspect='auto',
    extent=extent,
    cmap='RdBu_r'
)

In [ ]:
# =============================
# Group LRT maps + 2D cluster test
# =============================
from mne.stats import permutation_cluster_1samp_test

lrt_group_results = {}

if RUN_LRT_ALLTRIAL:
    for roi in LRT_ROIS:
        for predictor in LRT_PREDICTORS:
            lrt_maps, valid_subjects, fb_times, cue_times, skipped_reasons = compute_all_subjects_lrt_maps_from_npz(
                roi=roi,
                subjects=subjects,
                trial_save_dir=MV_TRIAL_SAVE_DIR,
                n_jobs=-1,
                verbose=True,
                dependent_var=predictor,
                aggregate_electrodes=True,
                min_valid_trials=MIN_TRIALS,
            )

            key = (roi, predictor)
            lrt_group_results[key] = {
                'lrt_maps': lrt_maps,
                'valid_subjects': valid_subjects,
                'feedback_times': fb_times,
                'cue_times': cue_times,
                'skipped_reasons': skipped_reasons,
                't_obs': None,
                'clusters': [],
                'cluster_pvals': np.array([]),
                'h0': np.array([]),
            }

            if lrt_maps is None or len(valid_subjects) < 2:
                print(f'Skipping cluster test for ROI={roi}, predictor={predictor}: insufficient subjects')
                continue

            t_obs, clusters, cluster_pvals, h0 = permutation_cluster_1samp_test(
                lrt_maps,
                n_permutations=LRT_CLUSTER_N_PERM,
                tail=LRT_CLUSTER_TAIL,
                out_type='mask',
                seed=LRT_CLUSTER_SEED,
                verbose=True,
            )

            lrt_group_results[key]['t_obs'] = t_obs
            lrt_group_results[key]['clusters'] = clusters
            lrt_group_results[key]['cluster_pvals'] = cluster_pvals
            lrt_group_results[key]['h0'] = h0

            out_npz = os.path.join(LRT_SAVE_DIR, f'mv_lrt_{roi}_{predictor}.npz')
            np.savez_compressed(
                out_npz,
                lrt_maps=lrt_maps,
                t_obs=t_obs,
                cluster_pvals=cluster_pvals,
                feedback_times=fb_times,
                cue_times=cue_times,
                valid_subjects=np.asarray(valid_subjects, dtype=object),
                skipped_subjects=np.asarray(list(skipped_reasons.items()), dtype=object),
                roi=np.asarray(roi),
                predictor=np.asarray(predictor),
                n_permutations=np.int32(LRT_CLUSTER_N_PERM),
                tail=np.int32(LRT_CLUSTER_TAIL),
                p_thresh=np.float32(LRT_CLUSTER_P_THRESH),
            )

            print(f'Saved LRT result: {out_npz}')
else:
    print('RUN_LRT_ALLTRIAL=False, skipping LRT group computation.')


In [ ]:
# load  the lrt_group_results

            

In [ ]:
LRT_ROIS = ['AMY']

In [ ]:
# Load lrt_group_results from saved NPZ files (LRT_SAVE_DIR)
# Run this cell instead of the Group LRT computation when results are already saved.
lrt_group_results = {}
for roi in LRT_ROIS:
    for predictor in LRT_PREDICTORS:
        out_npz = os.path.join(LRT_SAVE_DIR, f'mv_lrt_{roi}_{predictor}.npz')
        if not os.path.isfile(out_npz):
            print(f'Missing: {out_npz}')
            continue
        data = np.load(out_npz, allow_pickle=True)
        # Reconstruct dict expected by downstream (cluster masks not saved in npz)
        lrt_group_results[(roi, predictor)] = {
            'lrt_maps': data['lrt_maps'],
            't_obs': data['t_obs'],
            'cluster_pvals': data['cluster_pvals'] if 'cluster_pvals' in data else np.array([]),
            'clusters': [],  # not saved in npz; contour overlay will be omitted
            'feedback_times': data['feedback_times'],
            'cue_times': data['cue_times'],
            'valid_subjects': data['valid_subjects'] if 'valid_subjects' in data else None,
        }
        if 'skipped_subjects' in data:
            sk = data['skipped_subjects']
            lrt_group_results[(roi, predictor)]['skipped_reasons'] = dict(sk.tolist()) if sk.size else {}
        data.close()
print(f'Loaded lrt_group_results for {len(lrt_group_results)} (roi, predictor) pairs.')

In [ ]:
# =============================
# Plot LRT T_obs + significant clusters
# =============================
for (roi, predictor), res in lrt_group_results.items():
    lrt_maps = res.get('lrt_maps', None)
    t_obs = res.get('t_obs', None)
    clusters = res.get('clusters', [])
    cluster_pvals = np.asarray(res.get('cluster_pvals', np.array([])))
    fb_times = res.get('feedback_times', None)
    cue_times = res.get('cue_times', None)

    if lrt_maps is None or t_obs is None:
        print(f'No cluster-test output to plot for ROI={roi}, predictor={predictor}')
        continue

    fig, ax = plt.subplots(1, 1, figsize=(4, 4))
    # vmax = np.nanmax(np.abs(t_obs))
    # if not np.isfinite(vmax) or vmax == 0:
    #     vmax = 1.0

    ax.imshow(
        t_obs,
        aspect='auto',
        origin='lower',
        cmap='RdBu_r',
        interpolation='nearest',
        extent=[float(cue_times.min()), float(cue_times.max()), float(fb_times.min()), float(fb_times.max())],
    )

    n_sig = 0
    if len(clusters) > 0:
        for c_mask, pval in zip(clusters, cluster_pvals):
            if pval <= LRT_CLUSTER_P_THRESH:
                n_sig += 1
                ax.contour(
                    cue_times,
                    fb_times,
                    c_mask.astype(float),
                    levels=[0.5],
                    colors='black',
                    linewidths=1.5,
                    linestyles='--',
                )

    ax.set_title(f'T_obs LRT map | ROI={roi} | predictor={predictor}')
    ax.set_xlabel('Cue Time (s)')
    ax.set_ylabel('Feedback Time (s)')
    plt.tight_layout()
    plt.show()

    print(f'Found {len(clusters)} clusters, {n_sig} significant (p ≤ {LRT_CLUSTER_P_THRESH})')



In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(7, 5))

# Panel 1: Mean LRT map across subjects
ax.imshow(lrt_maps.mean(axis=0),
aspect='auto', origin='lower',
                    cmap='RdBu_r')
ax.set_xlabel('Cue Time')
ax.set_ylabel('Feedback Time')

In [ ]:
# =============================
# LRT output index
# =============================
print('LRT save dir:', LRT_SAVE_DIR)

rows = []
for (roi, predictor), res in lrt_group_results.items():
    valid_subjects = res.get('valid_subjects', [])
    skipped = res.get('skipped_reasons', {})
    n_clusters = len(res.get('clusters', [])) if res.get('clusters', None) is not None else 0
    pvals = np.asarray(res.get('cluster_pvals', np.array([])))
    n_sig = int(np.sum(pvals <= LRT_CLUSTER_P_THRESH)) if pvals.size > 0 else 0
    rows.append({
        'roi': roi,
        'predictor': predictor,
        'n_valid_subjects': len(valid_subjects),
        'n_skipped_subjects': len(skipped),
        'n_clusters': n_clusters,
        'n_sig_clusters': n_sig,
        'npz_path': os.path.join(LRT_SAVE_DIR, f'mv_lrt_{roi}_{predictor}.npz'),
    })

lrt_summary_df = pd.DataFrame(rows)
display(lrt_summary_df)

summary_csv = os.path.join(LRT_SAVE_DIR, 'mv_lrt_summary.csv')
lrt_summary_df.to_csv(summary_csv, index=False)
print('Saved LRT summary CSV:', summary_csv)


In [ ]:
# =============================
# Group-level 1D plot (Hits - Misses, averaged over encoding time)
# =============================
GROUP_ROI = 'ACC'
try:
    GROUP_CLUSTER_P_THRESH
except NameError:
    GROUP_CLUSTER_P_THRESH = 0.05

GROUP_CLUSTER_TEST_1D = True

diff_1d_list = []
included_subjects_1d = []
skipped_reasons_1d = {}

for subj in subjects:
    try:
        z_reactivation, feedback_times, cue_times, meta = load_trial_level(
            GROUP_ROI, subj, MV_TRIAL_SAVE_DIR, dependent_var=GROUP_PREDICTOR
        )
    except Exception:
        skipped_reasons_1d[subj] = 'missing_or_unreadable_npz'
        continue

    if z_reactivation.ndim != 3:
        skipped_reasons_1d[subj] = f'unexpected_ndim_{z_reactivation.ndim}'
        continue

    memory_outcomes = np.asarray(meta.get('memory_outcomes', np.array([], dtype=object)))
    if len(memory_outcomes) != z_reactivation.shape[-1]:
        skipped_reasons_1d[subj] = 'trial_mismatch'
        continue

    hits_mask = memory_outcomes == 'hits'
    misses_mask = memory_outcomes == 'misses'
    if hits_mask.sum() == 0 or misses_mask.sum() == 0:
        skipped_reasons_1d[subj] = 'missing_hits_or_misses'
        continue

    diff_map = (
        np.nanmean(z_reactivation[:, :, hits_mask], axis=2) -
        np.nanmean(z_reactivation[:, :, misses_mask], axis=2)
    )
    if np.all(~np.isfinite(diff_map)):
        skipped_reasons_1d[subj] = 'all_nan_diff_map'
        continue

    if diff_map.shape != (len(feedback_times), len(cue_times)):
        skipped_reasons_1d[subj] = 'diff_map_shape_mismatch'
        continue

    diff_1d = np.nanmean(diff_map, axis=0)
    diff_1d_list.append(diff_1d)
    included_subjects_1d.append(subj)

if len(diff_1d_list) > 0:
    D_1d = np.stack(diff_1d_list, axis=0)
    group_mean_1d = np.nanmean(D_1d, axis=0)
    group_se_1d = np.nanstd(D_1d, axis=0, ddof=1) / np.sqrt(D_1d.shape[0])
    group_se_1d = np.nan_to_num(group_se_1d, nan=0.0)

    clusters_1d = []
    cluster_pvals_1d = np.array([])
    sig_mask_1d = np.zeros(len(cue_times), dtype=bool)

    if GROUP_CLUSTER_TEST_1D and D_1d.shape[0] >= 2:
        try:
            T_obs, clusters_1d, cluster_pvals_1d, H0 = permutation_cluster_1samp_test(
                D_1d,
                n_permutations=5000,
                out_type='mask',
                seed=42,
                verbose=True,
            )
            if len(clusters_1d) > 0:
                for c_mask, pval in zip(clusters_1d, cluster_pvals_1d):
                    if pval <= GROUP_CLUSTER_P_THRESH:
                        sig_mask_1d |= c_mask.astype(bool)
        except Exception as e:
            print(f'1D cluster test failed: {e}')

    plt.figure(figsize=(8, 4))
    def _contiguous_runs(mask):
        runs = []
        i = 0
        while i < len(mask):
            if mask[i]:
                start = i
                while i < len(mask) and mask[i]:
                    i += 1
                runs.append((start, i - 1))
            else:
                i += 1
        return runs

    plt.plot(cue_times, group_mean_1d, color='C0', linewidth=1.5)
    plt.fill_between(cue_times, group_mean_1d - group_se_1d, group_mean_1d + group_se_1d, alpha=0.2, color='C0')

    if np.any(sig_mask_1d):
        for start, end in _contiguous_runs(sig_mask_1d):
            plt.axvspan(cue_times[start], cue_times[end], alpha=0.2, color='gray')

    plt.axhline(0, color='k', linestyle='--', linewidth=1)
    plt.title(f'Group mean (Hits - Misses), averaged over encoding time\nROI={GROUP_ROI} | predictor={GROUP_PREDICTOR}')
    plt.xlabel('Cue Time (s)')
    plt.ylabel('z-reactivation (mean over encoding)')
    sns.despine()
    plt.tight_layout()
    plt.show()

    if GROUP_CLUSTER_TEST_1D and D_1d.shape[0] >= 2:
        n_clusters = len(clusters_1d)
        sig_pvals = np.sort(np.asarray(cluster_pvals_1d)[np.asarray(cluster_pvals_1d) <= GROUP_CLUSTER_P_THRESH]) if n_clusters > 0 else np.array([])
        print(f'1D cluster test: n_clusters={n_clusters}, n_significant={len(sig_pvals)} (p<={GROUP_CLUSTER_P_THRESH})')
        if len(sig_pvals) > 0:
            print('Significant cluster p-values:', sig_pvals)
        else:
            print(f'No significant 1D clusters at p<={GROUP_CLUSTER_P_THRESH}.')
else:
    print('No subjects met inclusion criteria for 1D group-level plot.')

print(f'Included subjects: {len(included_subjects_1d)} / {len(subjects)}')
if len(included_subjects_1d) > 0:
    print('Included IDs:', included_subjects_1d)
if len(skipped_reasons_1d) > 0:
    print('Skipped reason counts:')
    print(pd.Series(list(skipped_reasons_1d.values())).value_counts())

In [ ]:
# # =============================
# # Glass brain: time-averaged activation per ROI (across subjects)
# # =============================
# # Load time-averaged activation patterns + MNI coords from all MV NPZ results
# activation_mni_df, glass_skipped = load_mv_activation_and_mni(
#     trial_save_dir=MV_TRIAL_SAVE_DIR,
#     base_dir=base_dir,
#     load_dir=load_dir,
#     rois=ROIS,
#     predictors=PREDICTORS,
#     subjects=subjects,
#     verbose=True,
# )
# if len(glass_skipped) > 0:
#     print(f"Skipped entries: {len(glass_skipped)}")
# print(f"Loaded {len(activation_mni_df)} rows (subject × roi × predictor × electrode with MNI)")

# # Load MNI template for interpolation grid (requires nilearn + nibabel)
# try:
#     from nilearn import datasets
#     from nilearn import plotting as nlp
#     template_path = datasets.load_mni152_template(resolution=2)
#     from nilearn.image import load_img
#     template_img = load_img(template_path)
# except ImportError as e:
#     print("Glass brain requires nilearn and nibabel. Install with: pip install nilearn nibabel")
#     raise e

# # One glass brain per ROI: interpolate scattered (MNI, activation) to heatmap, then plot
# SIGMA_MM = 10.0  # Gaussian smoothing for heatmap
# for roi in ROIS:
#     roi_df = activation_mni_df[activation_mni_df["roi"] == roi]
#     if roi_df.empty:
#         print(f"No data for ROI {roi}, skipping")
#         continue
#     mni_coords = roi_df[["mni_x", "mni_y", "mni_z"]].values
#     values = roi_df["time_avg_activation"].values
#     heatmap_img = coords_values_to_heatmap_nifti(
#         mni_coords, values, template_img, sigma_mm=SIGMA_MM
#     )
#     fig = nlp.plot_glass_brain(
#         heatmap_img,
#         display_mode="ortho",
#         colorbar=True,
#         title=f"Activation (time-avg) — {roi}",
#         cmap="cold_hot",
#         threshold="auto",
#         figure=plt.figure(figsize=(8, 6)),
#     )
#     plt.show()

In [ ]:
# =============================
# Group-level plots + stats
# =============================
from mne.stats import permutation_cluster_1samp_test

GROUP_ROI = 'OFC'
GROUP_PREDICTOR = PREDICTORS[0]
GROUP_CLUSTER_TEST_2D = True


group_diff_maps = []
included_subjects = []
skipped_reasons = {}

for subj in subjects:
    try:
        z_reactivation, feedback_times, cue_times, meta = load_trial_level(
            GROUP_ROI, subj, MV_TRIAL_SAVE_DIR, dependent_var=GROUP_PREDICTOR
        )
    except Exception:
        skipped_reasons[subj] = 'missing_or_unreadable_npz'
        continue

    if z_reactivation.ndim != 3:
        skipped_reasons[subj] = f'unexpected_ndim_{z_reactivation.ndim}'
        continue

    memory_outcomes = np.asarray(meta.get('memory_outcomes', np.array([], dtype=object)))
    if len(memory_outcomes) != z_reactivation.shape[-1]:
        skipped_reasons[subj] = 'trial_mismatch'
        continue

    hits_mask = memory_outcomes == 'hits'
    misses_mask = memory_outcomes == 'misses'
    if hits_mask.sum() == 0 or misses_mask.sum() == 0:
        skipped_reasons[subj] = 'missing_hits_or_misses'
        continue

    diff_map = (
        np.nanmean(z_reactivation[:, :, hits_mask], axis=2) -
        np.nanmean(z_reactivation[:, :, misses_mask], axis=2)
    )
    if np.all(~np.isfinite(diff_map)):
        skipped_reasons[subj] = 'all_nan_diff_map'
        continue

    if diff_map.shape != (len(feedback_times), len(cue_times)):
        skipped_reasons[subj] = 'diff_map_shape_mismatch'
        continue

    group_diff_maps.append(diff_map)
    included_subjects.append(subj)

if len(group_diff_maps) > 0:
    D = np.stack(group_diff_maps, axis=0)
    assert D.ndim == 3, f'Expected 3D stack, got shape {D.shape}'

    group_mean_map = np.nanmean(D, axis=0)
    vmax = np.nanmax(np.abs(group_mean_map))
    if not np.isfinite(vmax) or vmax == 0:
        vmax = 1.0

    clusters = []
    cluster_pvals = np.array([])
    sig_mask = np.zeros_like(group_mean_map, dtype=bool)

    if GROUP_CLUSTER_TEST_2D:
        if D.shape[0] < 2:
            print('Insufficient subjects for 2D cluster test (need >=2).')
        else:
            try:
                T_obs, clusters, cluster_pvals, H0 = permutation_cluster_1samp_test(
                    D,
                    n_permutations=5000,
                    out_type='mask',
                    tail=1,
                    seed=42,
                    verbose=True,
                )

                if len(clusters) > 0:
                    for c_mask, pval in zip(clusters, cluster_pvals):
                        if pval <= GROUP_CLUSTER_P_THRESH:
                            sig_mask |= c_mask.astype(bool)
            except Exception as e:
                print(f'2D cluster test failed: {e}')

    plt.figure(figsize=(6, 5))
    plt.imshow(
        group_mean_map,
        origin='lower',
        aspect='auto',
        extent=[float(cue_times.min()), float(cue_times.max()), float(feedback_times.min()), float(feedback_times.max())],
        cmap='RdBu_r',
        vmin=-vmax,
        vmax=vmax,
    )

    if np.any(sig_mask):
        plt.contour(
            cue_times,
            feedback_times,
            sig_mask.astype(int),
            levels=[0.5],
            colors='k',
            linewidths=1.5,
        )

    plt.title(f'Group mean (Hits - Misses)\nROI={GROUP_ROI} | predictor={GROUP_PREDICTOR}')
    plt.xlabel('Cue Time (s)')
    plt.ylabel('Feedback Time (s)')
    # plt.colorbar(shrink=0.8)
    plt.tight_layout()
    plt.show()

    if GROUP_CLUSTER_TEST_2D and D.shape[0] >= 2:
        n_clusters = len(clusters)
        sig_pvals = np.sort(np.asarray(cluster_pvals)[np.asarray(cluster_pvals) <= GROUP_CLUSTER_P_THRESH]) if n_clusters > 0 else np.array([])
        print(f'2D cluster test: n_clusters={n_clusters}, n_significant={len(sig_pvals)} (p<={GROUP_CLUSTER_P_THRESH})')
        if len(sig_pvals) > 0:
            print('Significant cluster p-values:', sig_pvals)
        else:
            print(f'No significant 2D clusters at p<={GROUP_CLUSTER_P_THRESH}.')
else:
    print('No subjects met inclusion criteria for group-level map.')

print(f'Included subjects: {len(included_subjects)} / {len(subjects)}')
if len(included_subjects) > 0:
    print('Included IDs:', included_subjects)
if len(skipped_reasons) > 0:
    print('Skipped reason counts:')
    print(pd.Series(list(skipped_reasons.values())).value_counts())


In [ ]:
cluster_pvals

In [ ]:
cluster_pvals

In [ ]:
# get the diagonal of group_mean_map
group_mean_map_diag = np.diag(group_mean_map)

# plot the diagonal of group_mean_map
plt.plot(group_mean_map_diag)
plt.show()


In [ ]:
# =============================
# Group mean z_reactivation (all trials)
# =============================
try:
    GROUP_CLUSTER_P_THRESH
except NameError:
    GROUP_CLUSTER_P_THRESH = 0.05

group_mean_maps = []
all_trials_included_subjects = []
all_trials_skipped_reasons = {}

for subj in subjects:
    try:
        z_reactivation, feedback_times, cue_times, meta = load_trial_level(
            GROUP_ROI, subj, MV_TRIAL_SAVE_DIR, dependent_var=GROUP_PREDICTOR
        )
    except Exception:
        all_trials_skipped_reasons[subj] = 'missing_or_unreadable_npz'
        continue

    if z_reactivation.ndim != 3:
        all_trials_skipped_reasons[subj] = f'unexpected_ndim_{z_reactivation.ndim}'
        continue

    mean_map = np.nanmean(z_reactivation, axis=2)
    if np.all(~np.isfinite(mean_map)):
        all_trials_skipped_reasons[subj] = 'all_nan_mean_map'
        continue

    if mean_map.shape != (len(feedback_times), len(cue_times)):
        all_trials_skipped_reasons[subj] = 'mean_map_shape_mismatch'
        continue

    group_mean_maps.append(mean_map)
    all_trials_included_subjects.append(subj)

if len(group_mean_maps) > 0:
    D = np.stack(group_mean_maps, axis=0)
    assert D.ndim == 3, f'Expected 3D stack, got shape {D.shape}'

    group_mean_map = np.nanmean(D, axis=0)
    vmax = np.nanmax(np.abs(group_mean_map))
    if not np.isfinite(vmax) or vmax == 0:
        vmax = 1.0

    clusters = []
    cluster_pvals = np.array([])
    sig_mask = np.zeros_like(group_mean_map, dtype=bool)

    if GROUP_CLUSTER_TEST_2D:
        if D.shape[0] < 2:
            print('Insufficient subjects for 2D cluster test (need >=2).')
        else:
            try:
                T_obs, clusters, cluster_pvals, H0 = permutation_cluster_1samp_test(
                    D,
                    n_permutations=1000,
                    tail=1,
                    threshold=0,
                    out_type='mask',
                    seed=42,
                    verbose=True,
                )

                if len(clusters) > 0:
                    for c_mask, pval in zip(clusters, cluster_pvals):
                        if pval <= GROUP_CLUSTER_P_THRESH:
                            sig_mask |= c_mask.astype(bool)
            except Exception as e:
                print(f'2D cluster test failed: {e}')

    plt.figure(figsize=(6, 5))
    plt.imshow(
        group_mean_map,
        origin='lower',
        aspect='auto',
        extent=[float(cue_times.min()), float(cue_times.max()), float(feedback_times.min()), float(feedback_times.max())],
        cmap='RdBu_r',
        vmin=-vmax,
        vmax=vmax,
    )

    if np.any(sig_mask):
        plt.contour(
            cue_times,
            feedback_times,
            sig_mask.astype(int),
            levels=[0.5],
            colors='k',
            linewidths=1.5,
        )

    plt.title(f'Group mean (all trials)\nROI={GROUP_ROI} | predictor={GROUP_PREDICTOR}')
    plt.xlabel('Cue Time (s)')
    plt.ylabel('Feedback Time (s)')
    plt.colorbar(shrink=0.8)
    plt.tight_layout()
    plt.show()

    if GROUP_CLUSTER_TEST_2D and D.shape[0] >= 2:
        n_clusters = len(clusters)
        sig_pvals = np.sort(np.asarray(cluster_pvals)[np.asarray(cluster_pvals) <= GROUP_CLUSTER_P_THRESH]) if n_clusters > 0 else np.array([])
        print(f'2D cluster test: n_clusters={n_clusters}, n_significant={len(sig_pvals)} (p<={GROUP_CLUSTER_P_THRESH})')
        if len(sig_pvals) > 0:
            print('Significant cluster p-values:', sig_pvals)
        else:
            print(f'No significant 2D clusters at p<={GROUP_CLUSTER_P_THRESH}.')
else:
    print('No subjects met inclusion criteria for group-level map (all trials).')

print(f'Included subjects: {len(all_trials_included_subjects)} / {len(subjects)}')
if len(all_trials_included_subjects) > 0:
    print('Included IDs:', all_trials_included_subjects)
if len(all_trials_skipped_reasons) > 0:
    print('Skipped reason counts:')
    print(pd.Series(list(all_trials_skipped_reasons.values())).value_counts())

## Group encoding plot (all subjects)

In [ ]:
# =============================
# Plot group encoding (all-subject)
# =============================
from mne.stats import permutation_cluster_1samp_test

if len(mv_encoding_results) == 0:
    print('No batch MV encoding results in memory. Run the batch encoding cell first, or load from CSV.')
else:
    for predictor in PREDICTORS:
        if predictor not in mv_encoding_results:
            csv_path = build_multivariate_csv_path(load_dir, BAND, EVENT_ENCODING, predictor, suffix='smoothed_final')
            if os.path.exists(csv_path):
                model_df = pd.read_csv(csv_path)
            else:
                print(f'Skipping {predictor}: no in-memory results and no CSV at {csv_path}')
                continue
        else:
            model_df = mv_encoding_results[predictor].copy()

        fig, ax = plt.subplots(2, 3, figsize=(20, 10), sharey=True)
        n_perm_cluster = 5000
        plot_col = 'z_metric' if ENCODING_USE_SURROGATES else 'metric'
        metric_names = model_df['metric_name'].dropna().astype(str).unique() if 'metric_name' in model_df.columns else np.array([])
        is_bal = np.any(metric_names == 'cv_balanced_accuracy')
        y_label = 'z(Balanced Accuracy vs surrogate)' if (ENCODING_USE_SURROGATES and is_bal) else ('z(R² vs surrogate)' if ENCODING_USE_SURROGATES else ('Balanced Accuracy' if is_bal else 'R²'))

        for region in ['OFC', 'ACC', 'HPC', 'dmPFC', 'dlPFC', 'AMY']:
            ix, jx = {
                'OFC': (0, 0),
                'ACC': (0, 1),
                'HPC': (0, 2),
                'dmPFC': (1, 0),
                'dlPFC': (1, 1),
                'AMY': (1, 2),
            }[region]
            color = {
                'HPC': np.array([0.44314, 0.2902, 1]),
                'AMY': np.array([0.49412, 0, 0.90196]),
                'OFC': np.array([1, 0.75686, 0.41569]),
                'ACC': np.array([0.7098, 0.27059, 0.68235]),
                'dmPFC': np.array([0.40784, 0.01961, 0]),
                'dlPFC': np.array([1, 0.06275, 0.06275]),
            }[region]

            region_data = model_df[
                (model_df['predictor'] == predictor) &
                (model_df['region'] == region)
            ].copy()

            if len(region_data) == 0:
                ax[ix, jx].set_title(f'{region} (no data)')
                ax[ix, jx].axis('off')
                continue

            unique_ts = np.sort(region_data['ts'].unique())
            mean_metric = region_data.groupby('ts')[plot_col].mean().reindex(unique_ts).to_numpy()
            se_metric = region_data.groupby('ts')[plot_col].sem().reindex(unique_ts).to_numpy()

            ax[ix, jx].plot(unique_ts, mean_metric, color=color, linewidth=1.5)
            ax[ix, jx].fill_between(
                unique_ts,
                mean_metric - np.nan_to_num(se_metric, nan=0.0),
                mean_metric + np.nan_to_num(se_metric, nan=0.0),
                color=color,
                alpha=0.2,
            )

            subj_time = region_data.pivot_table(index='subj', columns='ts', values=plot_col, aggfunc='mean')
            subj_time = subj_time.reindex(columns=unique_ts)
            subj_time = subj_time.dropna(axis=0, how='any')

            if subj_time.shape[0] >= 2:
                try:
                    T_obs, clusters, cluster_p_values, _ = permutation_cluster_1samp_test(
                        subj_time.to_numpy(dtype=float),
                        n_permutations=n_perm_cluster,
                        tail=1,
                        threshold=0,
                        verbose=True,
                    )
                    for i_c, c in enumerate(clusters):
                        if cluster_p_values[i_c] <= 0.05:
                            c_ix = c[0]
                            c_ts = unique_ts[c_ix]
                            if len(c_ts) > 0:
                                y0, y1 = ax[ix, jx].get_ylim()
                                band_lo = y0 + 0.88 * (y1 - y0)
                                band_hi = y0 + 0.96 * (y1 - y0)
                                ax[ix, jx].fill_between(c_ts, band_lo, band_hi, color=color, alpha=0.35)
                except Exception as e:
                    print(f'Cluster test failed for predictor={predictor}, region={region}: {e}')

            if EVENT_ENCODING in ['cue_on', 'choice_time']:
                xticks = unique_ts[::max(1, len(unique_ts)//5)]
            else:
                xticks = unique_ts[::max(1, len(unique_ts)//7)]

            ax[ix, jx].set(
                xticks=xticks,
                xticklabels=[f'{int(t)}' for t in xticks],
                xlim=[float(np.min(unique_ts) - 100), float(np.max(unique_ts) + 100)],
                xlabel='time (ms)',
                ylabel=y_label,
                title=f'{region}'
            )
            ax[ix, jx].axhline(0, color='black', linestyle='--', linewidth=1)
            sns.despine(ax=ax[ix, jx])

            if ix == 0:
                ax[ix, jx].set(xlabel=None, xticklabels=[])
            if jx > 0:
                ax[ix, jx].set(ylabel=None)

        plt.suptitle(
            f"Group MV encoding (pooled-alpha tuned) | predictor={predictor} | event={EVENT_ENCODING} | "
            f"metric={('cv_balanced_accuracy' if np.any(model_df.get('metric_name', pd.Series(dtype=str)) == 'cv_balanced_accuracy') else ENCODING_METRIC)} | surrogates={ENCODING_USE_SURROGATES}",
            y=1.02,
        )
        plt.tight_layout()
        plt.show()


In [ ]:
# from membandit.granger import #     load_behavior_tables, #     preprocess_learn_df, #     preprocess_combined_df, #

# learn_df_raw, combined_df_raw = load_behavior_tables(load_dir)
# learn_df = preprocess_learn_df(learn_df_raw)
# combined_df = preprocess_combined_df(combined_df_raw)

In [ ]:
sns.barplot(data=model_df.groupby(['subj', 'region']).mean(numeric_only=True).reset_index(), x='region', y='metric',
order=['dmPFC', 'OFC', 'ACC', 'dlPFC', 'HPC', 'AMY'], errorbar='se')
sns.stripplot(data=model_df.groupby(['subj', 'region']).mean(numeric_only=True).reset_index(), x='region', y='metric',
order=['dmPFC', 'OFC', 'ACC', 'dlPFC', 'HPC', 'AMY'])

In [ ]:
max_r2 = []
max_r2_time = []
alphas = []
betas = []
reward = []
dprime = []
for subj in region_data.subj.unique():
    # plot the ts and metric for the subject
    max_r2.append(region_data[region_data.subj == subj].metric.max())
    max_r2_time.append(region_data.iloc[region_data[region_data.subj == subj].metric.argmax()]['ts'])
    alphas.append(learn_df[learn_df.participant==subj].alpha.unique()[0])
    betas.append(learn_df[learn_df.participant==subj].beta.unique()[0])
    reward.append(learn_df[learn_df.participant==subj].reward.sum())
    dprime.append(combined_df[(combined_df.participant==subj) & (combined_df.condition=='Day1')].dprime.unique()[0])
    # plt.plot(region_data[region_data.subj == subj].ts, region_data[region_data.subj == subj].metric)
    # plt.show()




In [ ]:
sns.regplot(x=max_r2_time, y=reward)
pearsonr(max_r2_time, dprime)

- Does peak R^2 in a subject’s ROI correlate with learning rate? 

No

- Does peak R^2 in a subject’s ROI correlate with inverse temperature? 

No

- Does peak R^2 in a subject’s ROI correlate with reward? 

No

- Does peak R^2 in a subject’s ROI correlate with dprime?

No



The time, magnitude, and location of RPE decodability (given our electrode coverage) changes person-by-person. f

So, this analysis tells me two things: 

1. the regions that are encoding RPE information: OFC, ACC, dmPFC
2. the timing of maximal encoding: OFC (400, 800ms), ACC (400, 1400ms), and dmPFC (50, 700, 1300 ms)